# GPU-Only Training: RF, XGBoost, CatBoost, LightGBM

This notebook trains only four models with GPU acceleration and no cuML:
- Random Forest (GPU via XGBoost RF)
- XGBoost (GPU)
- CatBoost (GPU)
- LightGBM (GPU)

In [ ]:
# Optional: install missing packages in the active kernel
%pip install -q numpy pandas scipy scikit-learn xgboost catboost lightgbm

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from xgboost import XGBClassifier, XGBRFClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

print(f"Python executable: {sys.executable}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '<not set>')}")

try:
    smi = subprocess.check_output(['nvidia-smi'], text=True, stderr=subprocess.STDOUT)
    print("nvidia-smi detected.")
    print("\n".join(smi.splitlines()[:8]))
except Exception as e:
    raise RuntimeError(f"GPU check failed (nvidia-smi unavailable): {e}")

print("xgboost version:", __import__('xgboost').__version__)
print("catboost version:", __import__('catboost').__version__)
print("lightgbm version:", __import__('lightgbm').__version__)

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_CANDIDATES = [
    PROJECT_ROOT / 'merged_clinical_dietary_anthro_leftjoin.csv',
    PROJECT_ROOT.parent / 'merged_clinical_dietary_anthro_leftjoin.csv',
    PROJECT_ROOT / 'merged_clinical_dietary_leftjoin.csv',
    PROJECT_ROOT.parent / 'merged_clinical_dietary_leftjoin.csv',
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find a merged dataset CSV near this notebook.')

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape}")

lower_to_col = {c.lower(): c for c in df.columns}
target_candidates = ['target', 'label', 'outcome', 'hypertension', 'htn', 'class', 'y']
target_col = next((lower_to_col[c] for c in target_candidates if c in lower_to_col), None)

if target_col is None:
    sbp_col = next((c for c in df.columns if 'sbp' in c.lower() or 'systolic' in c.lower()), None)
    dbp_col = next((c for c in df.columns if 'dbp' in c.lower() or 'diastolic' in c.lower()), None)
    if sbp_col and dbp_col:
        target_col = '__derived_hypertension__'
        df[target_col] = ((df[sbp_col] >= 140) | (df[dbp_col] >= 90)).astype(int)
    else:
        raise ValueError('No binary target found and could not derive target from SBP/DBP columns.')

y = df[target_col]
X = df.drop(columns=[target_col])

if y.dtype == object:
    y = y.astype(str).str.strip().str.lower()
    map_bin = {'yes': 1, 'no': 0, 'true': 1, 'false': 0, '1': 1, '0': 0}
    if set(y.unique()).issubset(set(map_bin.keys())):
        y = y.map(map_bin)

y = pd.Series(y).astype(float)
if y.nunique(dropna=True) != 2:
    raise ValueError(f'Target must be binary. Found {y.nunique(dropna=True)} unique values in {target_col}.')

valid = y.notna()
X = X.loc[valid].copy()
y = y.loc[valid].astype(int)

X = X.replace([np.inf, -np.inf], np.nan)

numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print(f"Target column: {target_col}")
print(f"Samples: {len(X)}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

X_train_proc = preprocess.fit_transform(X_train)
X_test_proc = preprocess.transform(X_test)

if not sparse.issparse(X_train_proc):
    X_train_proc = sparse.csr_matrix(X_train_proc)
if not sparse.issparse(X_test_proc):
    X_test_proc = sparse.csr_matrix(X_test_proc)

print('Train matrix shape:', X_train_proc.shape)
print('Test matrix shape:', X_test_proc.shape)

In [ ]:
models = {
    'RandomForest_GPU_XGBRF': XGBRFClassifier(
        n_estimators=500,
        max_depth=10,
        learning_rate=1.0,
        subsample=0.8,
        colsample_bynode=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device='cuda',
        random_state=42,
        n_jobs=1
    ),
    'XGBoost_GPU': XGBClassifier(
        n_estimators=800,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device='cuda',
        random_state=42
    ),
    'CatBoost_GPU': CatBoostClassifier(
        iterations=800,
        depth=8,
        learning_rate=0.05,
        loss_function='Logloss',
        eval_metric='AUC',
        task_type='GPU',
        devices='0',
        random_seed=42,
        verbose=False
    ),
    'LightGBM_GPU': LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary',
        device_type='gpu',
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    print(f'\n=== Training {name} ===')
    t0 = time.time()
    try:
        model.fit(X_train_proc, y_train)

        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test_proc)[:, 1]
        else:
            y_prob = model.predict(X_test_proc)

        y_pred = (y_prob >= 0.5).astype(int)

        row = {
            'model': name,
            'status': 'ok',
            'train_seconds': round(time.time() - t0, 2),
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, y_prob),
        }
        results.append(row)
        trained_models[name] = model
        print('Done:', row)
    except Exception as e:
        row = {
            'model': name,
            'status': 'failed',
            'train_seconds': round(time.time() - t0, 2),
            'error': f'{type(e).__name__}: {e}'
        }
        results.append(row)
        print('Failed:', row)

results_df = pd.DataFrame(results)
results_df

In [ ]:
artifact_dir = Path.cwd() / 'gpu_only_tree_models_artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

results_path = artifact_dir / 'gpu_only_model_results.csv'
results_df.to_csv(results_path, index=False)
print(f'Saved results to: {results_path}')

ok_models = results_df[results_df['status'] == 'ok']['model'].tolist()
failed_models = results_df[results_df['status'] == 'failed']['model'].tolist()

print('Successful models:', ok_models)
print('Failed models:', failed_models)

if failed_models:
    print('\nAt least one GPU model failed. Check errors above for driver/runtime compatibility issues.')